# Stage 3 — CNN

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error
from matplotlib.lines import Line2D
from pathlib import Path

ROOT = Path("/Users/ewellmeyer/Documents/research/scripts/cloud_feedbacks")
CKPT = ROOT / "checkpoints" / "cnn"

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

# Pooled training set ground truth and CV predictions
y_pool       = np.load(CKPT / "y_pool.npy")
y_pool_cv    = np.load(CKPT / "y_pool_cv.npy")
n_ga8        = len(np.load(CKPT / "y_ga8.npy"))  # first n_ga8 entries are GA8

# Individual PPE arrays (for diagnostic scatter)
y_ga8        = np.load(CKPT / "y_ga8.npy")
y_ga9        = np.load(CKPT / "y_ga9.npy")
y_ga8_pred   = np.load(CKPT / "y_ga8_pred.npy")
y_ga9_pred   = np.load(CKPT / "y_ga9_pred.npy")

# OOS
y_c2         = np.load(CKPT / "y_c2.npy")
cfmip_fb     = np.load(CKPT / "cfmip_fb.npy")
cfmip_models = np.load(CKPT / "cfmip_models.npy", allow_pickle=True)
y_c2_pred    = np.load(CKPT / "y_c2_pred.npy")
y_cfmip_pred = np.load(CKPT / "y_cfmip_pred.npy")

# CERES
ceres_final  = float(np.load(CKPT / "ceres_final_pred.npy"))
ceres_folds  = np.load(CKPT / "ceres_fold_preds.npy")
N_FOLDS      = len(ceres_folds)

def scatter(ax, y_true, y_pred, label, color, marker="o", alpha=0.4, s=8):
    ax.scatter(y_true, y_pred, c=color, alpha=alpha, s=s, marker=marker)
    lo = min(y_true.min(), y_pred.min()) - 0.2
    hi = max(y_true.max(), y_pred.max()) + 0.2
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ax.set_title(f"{label}\nR²={r2:.3f}  RMSE={rmse:.2f} W/m²", fontsize=9)
    ax.set_xlabel("Actual δNetCRE (W/m²)")
    ax.set_ylabel("Predicted δNetCRE (W/m²)")
    return r2, rmse

print("Loaded.")

## Predicted vs actual

10-fold CV on the pooled GA8+GA9 training set (each member predicted by a model it was never trained on). The two PPE panels below that show the final model applied to GA8 and GA9 separately — these are **in-training-set** fits, not OOS, but useful to check for collapse or PPE-specific bias.

In [ ]:
# CV on pooled set
fig, ax = plt.subplots(figsize=(5, 5))
scatter(ax, y_pool, y_pool_cv, "GA8+GA9 pooled 10-fold CV", "steelblue")
plt.tight_layout()
plt.show()

# Final model fit on GA8 and GA9 individually (diagnostic — both in training set)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
scatter(axes[0], y_ga8, y_ga8_pred, "GA8 (in training set)", "steelblue")
scatter(axes[1], y_ga9, y_ga9_pred, "GA9 (in training set)", "darkorange")
plt.suptitle("Final model — training set fit by PPE generation", fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

## CFMIP structural generalisation test

Apply the final model to the 10 CFMIP models. These span completely different GCM families (GFDL, IPSL, MRI, …) so a high R² here means the CNN has learned something physically transferable, not just HadGEM-specific patterns. Stars mark individual models; the dashed 1:1 line and R²/RMSE are shown for context (small N=10, so treat as qualitative).

In [ ]:
PARENT_MODELS = {"HadGEM3-GC31-LL", "CESM2"}

fig, ax = plt.subplots(figsize=(6, 6))
for mod, yt, yp in zip(cfmip_models, cfmip_fb, y_cfmip_pred):
    is_parent = mod in PARENT_MODELS
    ax.scatter(yt, yp,
               c="crimson" if is_parent else "mediumpurple",
               marker="*" if is_parent else "o",
               s=120 if is_parent else 60, zorder=3)
    ax.annotate(mod, (yt, yp), fontsize=6.5,
                textcoords="offset points", xytext=(5, 2))

lo = min(cfmip_fb.min(), y_cfmip_pred.min()) - 0.3
hi = max(cfmip_fb.max(), y_cfmip_pred.max()) + 0.3
ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel("Actual δNetCRE (W/m²)")
ax.set_ylabel("Predicted δNetCRE (W/m²)")
ax.set_title("CFMIP structural generalisation (CNN)", fontsize=9)
ax.legend(handles=[
    Line2D([0],[0], marker="*", color="w", markerfacecolor="crimson",     markersize=10, label="PPE parent"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor="mediumpurple", markersize=8,  label="Other CFMIP"),
], fontsize=8)
plt.tight_layout()
plt.show()

print("\nCFMIP individual predictions:")
for mod, yt, yp in zip(cfmip_models, cfmip_fb, y_cfmip_pred):
    print(f"  {mod:<22s}  actual={yt:+.3f}  pred={yp:+.3f}  err={yp-yt:+.3f} W/m²")

## Summary metrics and CERES observational constraint

Collect all R²/RMSE numbers into one table mirroring the Stage 2 baseline, then apply the model to the CERES observation to get the constrained estimate of real-world cloud feedback.

The CERES prediction is a point estimate from a model trained on PPE members — use the spread across CV fold models as an uncertainty range.

In [ ]:
rows = [
    ("GA8+GA9 pooled 10-fold CV", y_pool,   y_pool_cv),
    ("CESM2 (SW proxy)",          y_c2,     y_c2_pred),
    ("CFMIP structural",          cfmip_fb, y_cfmip_pred),
]

print(f"{'Dataset':<30}  {'N':>4}  {'R²':>7}  {'RMSE (W/m²)':>12}")
print("-" * 60)
for label, yt, yp in rows:
    r2   = r2_score(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    print(f"{label:<30}  {len(yt):>4d}  {r2:>7.3f}  {rmse:>12.3f}")

print()
print("Ridge baseline (Stage 2, GA8 only):")
print("  GA8 CV   R²=0.871  RMSE=0.207 W/m²")
print("  GA9 OOS  R²=0.404  RMSE=0.569 W/m²")

print(f"\nCERES constraint (final model): {ceres_final:+.3f} W/m²  ({ceres_final/4:+.3f} W/m²/K)")
print(f"  Fold spread: {ceres_folds.min():+.3f} – {ceres_folds.max():+.3f} W/m²")
print(f"  Mean ± std:  {ceres_folds.mean():+.3f} ± {ceres_folds.std():.3f} W/m²")

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, N_FOLDS + 1), ceres_folds, color="steelblue", alpha=0.7)
ax.axhline(ceres_folds.mean(), color="k",         lw=1.2, ls="--",
           label=f"fold mean = {ceres_folds.mean():.3f}")
ax.axhline(ceres_final,        color="darkorange", lw=1.2, ls=":",
           label=f"final model = {ceres_final:.3f}")
ax.set_xlabel("CV fold model")
ax.set_ylabel("δNetCRE prediction (W/m²)")
ax.set_title("CERES observational constraint — CNN fold spread")
ax.set_xticks(range(1, N_FOLDS + 1))
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()